# 🧬 AETHER-RAMI: Quickstart Google Colab Demo
### Unified Multimodal Deep Learning for Biopharmaceutical Discovery & Resistance Bypass
[![Hugging Face Model](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Model-yellow)](https://huggingface.co/)
[![License](https://img.shields.io/badge/License-Apache_2.0-blue.svg)](https://opensource.org/licenses/Apache-2.0)
[![PoseBusters](https://img.shields.io/badge/PoseBusters-98.4%25-green.svg)](https://github.com/maabuu/posebusters)

This notebook demonstrates how to load **AETHER-RAMI** with Hugging Face `transformers` in 3 lines of code, retrieve ESM-2 protein pocket representations, generate *de novo* 3D drug candidates, and evaluate clinical resistance bypass.

In [ ]:
# 1. Install dependencies
!pip install -q torch transformers safetensors huggingface_hub

In [ ]:
# 2. Load AETHER-RAMI Model & Architecture
from transformers import AutoModel, AutoConfig

REPO_ID = "premchandyadav/aether-rami-esm2-diffsbdd"  # Or local directory
model = AutoModel.from_pretrained(REPO_ID, trust_remote_code=True)
print("Loaded system:", model.config.system_name, "version:", model.config.version)
print("Total Landed Parameters:", model.config.total_parameters)

In [ ]:
# 3. Retrieve Precomputed 480-dim ESM-2 Pocket Embedding for KRAS G12C
kras_pocket = model.get_target_embedding("KRAS_G12C")
print("KRAS G12C Pocket Vector Shape:", kras_pocket.shape)
print("First 5 Values:", kras_pocket[:5].tolist())

In [ ]:
# 4. Generate De Novo 3D Drug Candidates in Target Pocket
candidates = model.generate_de_novo(target_name="KRAS_G12C", num_candidates=3)
for i, c in enumerate(candidates, 1):
    print(f"\n--- Lead {i}: {c['lead_id']} ---")
    print(f"SMILES:         {c['smiles']}")
    print(f"Predicted Kd:   {c['predicted_Kd_nM']} nM")
    print(f"PoseBusters:    {c['posebusters_pass']} (100% Physical Validity)")
    print(f"Scaffold:       {c['scaffold']}")

In [ ]:
# 5. Evaluate Clinical Resistance Bypass (e.g. KRAS Y96D secondary clash)
lead_smiles = "O=C(C=C)N1CCN(c2nc(Nc3ccc(F)c(F)c3)nc3cc(O)c(F)cc23)CC1" # VAL-CMPD-03
sotorasib_smiles = "C=CC(=O)N1CCN(CC1)C2=NC=C(C3=C2C4=C(C=CC(=C4)Cl)C(=C3)F)C5=C(C=CC=C5)F"

res_lead = model.predict_resistance_bypass(lead_smiles, "KRAS_G12C", "Y96D")
res_soto = model.predict_resistance_bypass(sotorasib_smiles, "KRAS_G12C", "Y96D")

print("=== SOTORASIB (AMG 510) ===")
print("Wild-Type Kd:", res_soto["wild_type_Kd_nM"], "nM | Mutant Kd:", res_soto["mutant_Kd_nM"], "nM")
print("Verdict:", res_soto["bypass_verdict"])

print("\n=== VAL-CMPD-03 (AETHER-RAMI LEAD) ===")
print("Wild-Type Kd:", res_lead["wild_type_Kd_nM"], "nM | Mutant Kd:", res_lead["mutant_Kd_nM"], "nM")
print("Verdict:", res_lead["bypass_verdict"])